# Train Universal RF-DETR Colony Detector (Joint AGAR + 22022540 Datasets)

This notebook trains a single unified **RF-DETR Medium** detector combining:
1. **AGAR Dataset** (`anilanna4678/agar-dataset`): 18,000 real agar plate photos with diverse lighting, media, and agar variations.
2. **22022540 Dataset** (`anilanna4678/colony-dataset-22022540`): 369 high-res plates across 24 microbial species and 56,865 annotations in COCO format.

### Settings:
- **Accelerator**: `GPU T4 x2` (or GPU P100)
- **Internet**: `On`
- **Datasets**: Both `anilanna4678/agar-dataset` and `anilanna4678/colony-dataset-22022540` attached in Input section.

In [ ]:
# 1. GPU Check & Setup Workspace
import torch, os, sys, shutil, glob
assert torch.cuda.is_available(), 'GPU not found! Enable GPU T4 in Settings panel.'
print('GPU Device:', torch.cuda.get_device_name(0))
OUT = '/kaggle/working'

# Clone latest colony_detector repository
!rm -rf /kaggle/temp/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /kaggle/temp/colony_detector
%cd /kaggle/temp/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml
print('Setup complete.')

In [ ]:
# 2. Locate and Prepare Both Datasets in /kaggle/input
import glob, os, json, random, zipfile
from pathlib import Path

# --- A. Prepare AGAR Dataset Sample ---
candidates = glob.glob('/kaggle/input/**/dataset', recursive=True) + \
             glob.glob('/kaggle/input/**/raw', recursive=True) + \
             glob.glob('/kaggle/input/**/agar-dataset', recursive=True)

agar_dir = None
for c in candidates:
    if len(glob.glob(f'{c}/*.json')) > 0 or len(glob.glob(f'{c}/**/*.json', recursive=True)) > 0:
        agar_dir = c
        break

assert agar_dir is not None, f'AGAR dataset not found! Available inputs: {os.listdir("/kaggle/input")}'
print('Found AGAR dataset at:', agar_dir)

sample_agar_dir = Path('/kaggle/temp/agar_sample')
sample_agar_dir.mkdir(parents=True, exist_ok=True)

all_jsons = [p for p in (glob.glob(f'{agar_dir}/*.json') + glob.glob(f'{agar_dir}/**/*.json', recursive=True))
             if not p.endswith('annotations.json')]

countable, empty = [], []
for jp in all_jsons:
    img = Path(jp).parent / f'{Path(jp).stem}.jpg'
    if not img.exists():
        continue
    try:
        with open(jp, 'r') as f:
            d = json.load(f)
        n = d.get('colonies_number', len(d.get('labels', [])))
        if n == 0:
            empty.append((img, jp))
        elif 1 <= n <= 300:
            countable.append((img, jp))
    except Exception:
        continue

random.seed(42)
selected_agar = random.sample(countable, min(1200, len(countable))) + random.sample(empty, min(100, len(empty)))
for img, jp in selected_agar:
    shutil.copy2(img, sample_agar_dir / img.name)
    shutil.copy2(jp, sample_agar_dir / Path(jp).name)
print(f'Prepared {len(selected_agar)} balanced AGAR plates.')

# --- B. Prepare 22022540 Dataset ---
target_2202 = Path('/kaggle/temp/dataset_22022540')
target_2202.mkdir(parents=True, exist_ok=True)

zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
main_zip = next((z for z in zip_candidates if 'dataset.zip' in z or '22022540' in z), None)

if main_zip and not (target_2202 / 'annot_COCO.json').exists():
    print(f'Extracting {main_zip} to {target_2202}...')
    with zipfile.ZipFile(main_zip, 'r') as zf:
        zf.extractall(target_2202)
    data_2202 = target_2202
else:
    coco_files = glob.glob('/kaggle/input/**/annot_COCO.json', recursive=True)
    data_2202 = Path(coco_files[0]).parent if coco_files else target_2202

jpgs_2202 = list(data_2202.rglob('*.jpg'))
print(f'Prepared 22022540 dataset: {len(jpgs_2202)} plates from {data_2202}')

In [ ]:
# 3. Joint Tiling: Slicing Both Datasets into Unified Tiles
tile_out = '/kaggle/temp/tiles'
!rm -rf {tile_out}

# Slice both AGAR sample and 22022540 datasets into one unified COCO tile dataset
!python tools/build_training_set.py \
    --dataset {sample_agar_dir} {data_2202} \
    --tile 640 \
    --overlap 0.25 \
    --coco \
    --out {tile_out} 2>&1 | tee /kaggle/working/tile_log.txt

print('Unified tiling finished!')
!ls -la {tile_out}/coco

In [ ]:
# 4. Train Unified RF-DETR Medium Model (10 Epochs)
OUT = '/kaggle/working'

!python tools/train_rfdetr.py \
    --dataset /kaggle/temp/tiles/coco \
    --size medium \
    --resolution 640 \
    --epochs 10 \
    --batch 4 \
    --grad-accum 4 \
    --lr 1e-4 \
    --output {OUT}/rfdetr \
    --out {OUT}/rfdetr_tiles.pth \
    2>&1 | grep -E --line-buffered "Epoch|Best|mAP|Error|Traceback|->" | tee {OUT}/train_log.txt

print('Training complete! Best EMA weights saved to', f'{OUT}/rfdetr_tiles.pth')

In [ ]:
# 5. Calibrate Cut-offs on Validation Plates
import os
OUT = '/kaggle/working'
os.environ['CD_RFDETR_WEIGHTS'] = f'{OUT}/rfdetr_tiles.pth'
os.environ['CD_ENGINES'] = 'rfdetr'

!python tools/tune_detector.py \
    --dataset {data_2202} \
    --engine rfdetr \
    --split valid \
    --conf-only \
    --cache /kaggle/temp/valid_dets.pkl | tee {OUT}/tune_log.txt

!cp models/detector.json {OUT}/detector.json
print('\nTuned detector configuration:')
!cat {OUT}/detector.json

print('\nFinal generated files in output:')
!ls -lh {OUT}